# 09 MISSING: treatment of values blanked by the validity specification

Setting blanked values to zero is questionable for catenary voltage and current, since zero is a physically meaningful reading (a de-energised line). This notebook

1. counts the blanked values in June by channel, in the positive windows and in the test partition;
2. refits the primary specification with two alternatives to zero: the models' own missing-value handling (XGBoost and CatBoost) and forward fill, which carries the last valid reading forward and so uses no future information (all three models);
3. compares the scores with zero fill, and with the spread across seeds from `08_SEEDS`.

All fits use default class weighting and seed 42. The model code is the same as in `07_CLASSIFY`. Outputs: `OUTPUT/v4/rev_missing_counts.csv`, `OUTPUT/v4/rev_missing_results.csv`.

Paper: Section 2.7.1.

In [ ]:
import os, sys, time, warnings
import numpy as np, pandas as pd
import xgboost, catboost
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
warnings.filterwarnings("ignore")

A2 = "../ANALYSIS2"
env = os.environ.get
CFG = dict(
    labelled   = os.environ.get("REV_BUILDDIR", "build") + "/telemetry_labelled_v4.pkl",
    ref_pooled = f"{A2}/OUTPUT/classification_results_v3.csv",
    seeds_summary = env("REV_OUTDIR", "OUTPUT") + "/v4" + "/rev_seeds_summary.csv",
    outdir     = env("REV_OUTDIR", "OUTPUT") + "/v4",
    horizons   = env("REV_HORIZONS", "D5,W5,W60,W300").split(","),
    models     = env("REV_MODELS", "XGBoost,CatBoost,Logistic Regression").split(","),
    patience   = 100, test_frac = 0.20, val_frac = 0.20, seed = 42,
)
NUMERIC = ["KMH", "DIST_M", "KV", "AMP", "KWH", "BV", "PRESSURE", "TRACTION_PCT", "BG1_KN", "BG2_KN", "BG1_DCLV", "BG2_DCLV"]
WHEEL_SLIP = ["Wheel_Slip_TM2", "Wheel_Slip_TM3", "Wheel_Slip_TM4", "Wheel_Slip_TM5", "Wheel_Slip_TM6"]
PRIMARY = NUMERIC + WHEEL_SLIP
os.makedirs(CFG["outdir"], exist_ok=True)
print("Python", sys.version.split()[0], "| pandas", pd.__version__, "| xgboost", xgboost.__version__, "| catboost", catboost.__version__)

## 1. Blanked values in June

In [ ]:
tel = pd.read_pickle(CFG["labelled"]).sort_values("DATETIME").reset_index(drop=True)
if env("REV_SAMPLE"):                     # trial runs only
    tel = tel.iloc[:int(len(tel) * float(env("REV_SAMPLE")))].copy()
    print(f"TRIAL RUN on {len(tel):,} rows")
na = tel[PRIMARY].isna()
counts = pd.DataFrame({"blanked_values": na.sum()})
for h in ["D5", "W5", "W60", "W300"]:
    counts[f"in_{h}_positives"] = na[tel[h] == 1].sum()
counts = counts[counts.blanked_values > 0]
counts.loc["all channels"] = counts.sum()
print(f"{len(tel):,} June rows; rows with any blanked value: {int(na.any(axis=1).sum())}")
print(counts.to_string())
counts.to_csv(f"{CFG['outdir']}/rev_missing_counts.csv")

## 2. Model code (as in `07_CLASSIFY`)

Only the default-weighting branch is needed. CatBoost treats a missing value as smaller than any observed value (`nan_mode = "Min"`, its default); XGBoost learns a default direction at each split.

In [ ]:
def chrono_cut(y, frac):
    n = len(y); i = int(n * (1 - frac))
    while i < n - 1 and y[i] == 1 and y[i - 1] == 1:
        i += 1
    return i

def three_way_split(y, cfg):
    n = len(y); t = int(n * (1 - cfg["test_frac"]))
    while t < n - 1 and y[t] == 1:
        t += 1
    v = chrono_cut(y[:t], cfg["val_frac"])
    return np.arange(v), np.arange(v, t), np.arange(t), np.arange(t, n)

def make_model(name, seed, y, n_iter=None):
    if name == "Logistic Regression":
        return LogisticRegression(max_iter=2000, random_state=seed, class_weight="balanced")
    if name == "XGBoost":
        spw = float((y == 0).sum() / max((y == 1).sum(), 1))
        kw = dict(n_estimators=600, early_stopping_rounds=CFG["patience"]) if n_iter is None else dict(n_estimators=n_iter)
        return XGBClassifier(learning_rate=0.05, max_depth=6, subsample=0.8, colsample_bytree=0.8,
                             eval_metric="aucpr", tree_method="hist", scale_pos_weight=spw,
                             random_state=seed, n_jobs=-1, **kw)
    if name == "CatBoost":
        kw = dict(iterations=800, early_stopping_rounds=CFG["patience"]) if n_iter is None else dict(iterations=n_iter)
        return CatBoostClassifier(learning_rate=0.05, depth=6, eval_metric="PRAUC",
                                  auto_class_weights="Balanced", random_seed=seed, verbose=0, **kw)

def fit_score(name, X, y, seed):
    ia, iv, itr, ite = three_way_split(y, CFG)
    if name == "Logistic Regression":
        sc = StandardScaler().fit(X[itr])
        m = make_model(name, seed, y[itr]).fit(sc.transform(X[itr]), y[itr])
        s, n_iter = m.predict_proba(sc.transform(X[ite]))[:, 1], None
    else:
        m1 = make_model(name, seed, y[ia])
        if name == "XGBoost":
            m1.fit(X[ia], y[ia], eval_set=[(X[iv], y[iv])], verbose=False)
            n_iter = int(m1.best_iteration) + 1
        else:
            m1.fit(X[ia], y[ia], eval_set=(X[iv], y[iv]), use_best_model=True)
            n_iter = int(m1.get_best_iteration()) + 1
        m2 = make_model(name, seed, y[itr], n_iter=n_iter).fit(X[itr], y[itr])
        s = m2.predict_proba(X[ite])[:, 1]
    base = y[ite].mean()
    ap = average_precision_score(y[ite], s)
    return dict(pr_auc=ap, baseline=base, lift=ap / base, n_iter=n_iter,
                test_rows_blanked=int(np.isnan(X[ite]).any(axis=1).sum()) if np.isnan(X).any() else None)

## 3. Fits

Three versions of the input matrix, built over the whole month in time order before any rows are dropped:

* **zero**: as in the paper (`fillna(0)`);
* **native**: blanked values left missing (XGBoost and CatBoost only);
* **forward fill**: the last valid reading carried forward; a value with no earlier reading (none in practice) falls back to zero.

In [ ]:
raw = tel[PRIMARY].astype("float32")
XS = {"zero": raw.fillna(0).values, "native": raw.values, "forward fill": raw.ffill().fillna(0).values}
TREAT = {"XGBoost": ["zero", "native", "forward fill"], "CatBoost": ["zero", "native", "forward fill"], "Logistic Regression": ["zero", "forward fill"]}

rows = []
t_all = time.time()
for h in CFG["horizons"]:
    y0 = tel[h].astype(int).values
    keep = y0 != -1
    y = y0[keep]
    for mname in CFG["models"]:
        for tr in TREAT[mname]:
            t0 = time.time()
            r = fit_score(mname, XS[tr][keep], y, CFG["seed"])
            r.update(horizon=h, model=mname, treatment=tr, seconds=round(time.time() - t0, 1))
            rows.append(r)
            print(f"{h:5s} {mname:19s} {tr:12s} PR-AUC {r['pr_auc']:.4f}  iters {r['n_iter']}  {r['seconds']:.0f} s", flush=True)
res = pd.DataFrame(rows)
res.to_csv(f"{CFG['outdir']}/rev_missing_results.csv", index=False)
print(f"[{(time.time() - t_all) / 60:.1f} min]")

## 4. Comparison

In [ ]:
ref = (pd.read_csv(CFG["ref_pooled"]).query("sampling == 'Default (class weight)'").set_index(["horizon", "model"])["pr_auc"])
z = res[res.treatment == "zero"].copy()
z["reference"] = [ref.get((h, m), np.nan) for h, m in zip(z.horizon, z.model)]
z["reproduced"] = (z.pr_auc - z.reference).abs() < 1e-9
print(z[["horizon", "model", "pr_auc", "reference", "reproduced"]].round(4).to_string(index=False))
if True:   # corrected log: comparison with the submitted results only
    print("corrected log: results of the submitted manuscript shown for comparison only")
else:
    assert z.reproduced.all(), "zero-fill does not reproduce the submitted figures; stop and investigate"

t = res.pivot_table(index=["model", "treatment"], columns="horizon", values="pr_auc")
t = t.reindex(columns=[h for h in ["D5", "W5", "W60", "W300"] if h in t.columns])
print("\n--- PR-AUC by treatment"); print(t.round(4).to_string())

d = t.sub(t.xs("zero", level="treatment"), level="model").drop(index="zero", level="treatment")
print("\n--- change against zero-fill"); print(d.round(4).to_string())

if os.path.exists(CFG["seeds_summary"]):
    sd = (pd.read_csv(CFG["seeds_summary"]).query("target == 'pooled' and sampling == 'Default (class weight)'").pivot(index="model", columns="horizon", values="sd").reindex(columns=t.columns))
    print("\n--- for scale: SD of PR-AUC over five seeds (08_SEEDS)"); print(sd.round(4).to_string())

___
**END**